# Re-ranking with Sentence Transformers

In [ ]:
documents = [
    "This is a list which containing sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings.",
    "Understanding document structure aids in keyword extraction.",
    "Efficient keyword extraction enhances search accuracy.",
    "Semantic similarity improves document retrieval performance.",
    "Machine learning algorithms can optimize keyword extraction methods."
]

In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
# Load pre-trained Sentence Transformer model
model_name = 'sentence-transformers/paraphrase-xlm-r-multilingual-v1'

In [ ]:
model = SentenceTransformer(model_name)

In [ ]:
documents

In [ ]:
len(documents)

In [ ]:
document_embeddings = model.encode(documents)

In [ ]:
len(document_embeddings)

In [ ]:
for i, embedding in enumerate(document_embeddings):
    print(f"Document {i+1} embedding: {embedding}")

In [ ]:
query = "Natural language processing techniques enhance keyword extraction efficiency."

In [ ]:
query_embedding = model.encode(query)

In [ ]:
print("Query embedding:", query_embedding)

In [ ]:
len(query_embedding)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarities = cosine_similarity(np.array([query_embedding]), document_embeddings)

In [ ]:
similarities

In [ ]:
most_similar_index = np.argmax(similarities)

In [ ]:
most_similar_index

In [ ]:
most_similar_document = documents[most_similar_index]

In [ ]:
most_similar_document

In [ ]:
query

In [ ]:
similarity_score = similarities[0][most_similar_index]

In [ ]:
similarity_score

In [ ]:
sorted_indices = np.argsort(similarities[0])[::-1]

In [ ]:
ranked_documents = [(documents[i], similarities[0][i]) for i in sorted_indices]

In [ ]:
print("Ranked Documents:")
for rank, (document, similarity) in enumerate(ranked_documents, start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

In [ ]:
print("Top 4 Documents:")
for rank, (document, similarity) in enumerate(ranked_documents[:4], start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

In [ ]:
query

In [ ]:
!pip install rank_bm25

In [ ]:
from rank_bm25 import BM25Okapi

In [ ]:
top_4_documents = [doc[0] for doc in ranked_documents[:4]]

In [ ]:
top_4_documents

In [ ]:
tokenized_top_4_documents = [doc.split() for doc in top_4_documents]

In [ ]:
tokenized_top_4_documents

In [ ]:
tokenized_query = query.split()

In [ ]:
tokenized_query

In [ ]:
bm25=BM25Okapi(tokenized_top_4_documents)

In [ ]:
bm25

In [ ]:
bm25_scores = bm25.get_scores(tokenized_query)

In [ ]:
bm25_scores

In [ ]:
sorted_indices2 = np.argsort(bm25_scores)[::-1]

In [ ]:
sorted_indices2

In [ ]:
reranked_documents = [(top_4_documents[i], bm25_scores[i]) for i in sorted_indices2]

In [ ]:
reranked_documents

In [ ]:
print("Rerank of top 4 Documents:")
for rank, (document, similarity) in enumerate(reranked_documents, start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

In [ ]:
ranked_documents[:4]

# Cross Encoder

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [ ]:
pairs = []
for doc in top_4_documents:
    pairs.append([query, doc])

In [ ]:
scores = cross_encoder.predict(pairs)
scores

In [ ]:
scored_docs = zip(scores, top_4_documents)

In [ ]:
scored_docs

In [ ]:
reranked_document_cross_encoder = sorted(scored_docs, reverse=True)

In [ ]:
reranked_document_cross_encoder

# BM_25

In [ ]:
!pip install cohere

In [ ]:
import cohere
co = cohere.Client("nbDqU1hTVxWmXGbLYI6OnYhp4Cx40MZ5hOmO5oKX")

In [ ]:
response = co.rerank(
    model="rerank-english-v3.0",
    query="Natural language processing techniques enhance keyword extraction efficiency.",
    documents=top_4_documents,
    return_documents=True
)

In [ ]:
print(response)

In [ ]:
response.results[0].document.text

In [ ]:
response.results[0].relevance_score

In [ ]:
for i in range(4):
  print(f'text: {response.results[i].document.text} score: {response.results[i].relevance_score}')
